In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
from pathlib import Path

Path("../data").mkdir(exist_ok=True)

conn = sqlite3.connect("../db.sqlite3")

query = """
SELECT
    oi.id        AS "Order Item ID",
    oi.quantity  AS "Quantity",
    p.id         AS "Product ID",
    p.name       AS "Product Name",
    p.price      AS "Product Price",
    o.id         AS "Order ID",
    o.datetime   AS "Order Datetime"
FROM restaurant_orderitem AS oi
JOIN restaurant_product AS p ON oi.product_id = p.id
JOIN restaurant_order AS o ON oi.order_id = o.id
"""

pd.read_sql_query(query, conn).to_csv("../data/restaurant_data.csv", index=False)
conn.close()

df = pd.read_csv("../data/restaurant_data.csv")
print(df.shape)

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_qty = df.groupby("Product Name")["Quantity"].sum().nlargest(10)

top_qty.plot.pie(autopct="%1.1f%%", startangle=90, figsize=(7, 7),
                 title="Top 10 products by quantity sold", ylabel="")

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["Item Price"] = df["Product Price"] * df["Quantity"]

top_money = df.groupby("Product Name")["Item Price"].sum().nlargest(10)

top_money.plot.pie(autopct="%1.1f%%", startangle=90, figsize=(7, 7),
                   title="Top 10 products by revenue", ylabel="")

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["Order Datetime"] = pd.to_datetime(df["Order Datetime"])
df["Order Hour"] = df["Order Datetime"].dt.hour

income_by_hour = (
    df.groupby("Order Hour")["Item Price"]
    .sum()
    .reindex(range(24), fill_value=0)
)

income_by_hour.plot.bar(figsize=(10, 5), title="Total income by order hour",
                        xlabel="Order hour", ylabel="Total income")

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

df["Order Day Of The Week"] = df["Order Datetime"].dt.day_name()

income_by_day = (
    df.groupby("Order Day Of The Week")["Item Price"]
    .sum()
    .reindex(days, fill_value=0)
)

income_by_day.plot.bar(figsize=(10, 5), title="Total income by day of the week",
                       xlabel="Day of the week", ylabel="Total income", rot=45)